In [ ]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.49
    Uninstalling langchain-core-0.3.49:
      Successfully uninstalled langchain-core-0.3.49
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.7
    Uninstalling langchain-text-splitters-0.3.7:
      Successfully uninstalled langchain-text-splitters-0.3.7
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.22
    Uninstalling langchain-0.3.22:
      Successfully uninstalled langchain-0.3.22


In [ ]:
pip install youtube-transcript-api langchain faiss-cpu tiktoken openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.0 MB/s eta 0:00:00


In [ ]:
import os
from youtube_transcript_api import YouTubeTranscriptApi
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Step 1: Set OpenRouter API
os.environ["OPENAI_API_KEY"] = "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"  # Important: OpenRouter uses OpenAI client libraries
OPENAI_API_BASE = "https://openrouter.ai/api/v1"

# Step 2: Get YouTube Transcript
def get_youtube_transcript(video_id):
    transcript = YouTubeTranscriptApi.get_transcript(video_id)
    text = " ".join([entry['text'] for entry in transcript])
    return text

# Step 3: Split Text
def split_text(text, chunk_size=1000, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_text(text)
    return chunks

# Step 4: Build Retriever
def build_retriever(chunks):
    embeddings = OpenAIEmbeddings(openai_api_base=OPENAI_API_BASE)
    vectorstore = FAISS.from_texts(chunks, embedding=embeddings)
    return vectorstore.as_retriever()

# Step 5: Create RAG Chain
def create_rag_chain(retriever, model="openai/gpt-3.5-turbo"):
    llm = ChatOpenAI(
        temperature=0,
        openai_api_base=OPENAI_API_BASE,
        model=model,
    )
    prompt = PromptTemplate(
        template="Given the following context, summarize the key points of the video:\n\n{context}\n\nSummary:",
        input_variables=["context"]
    )
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type_kwargs={"prompt": prompt}
    )
    return qa_chain

# Step 6: Full Pipeline
def summarize_youtube_video(video_url, model="openai/gpt-3.5-turbo"):
    video_id = video_url.split("v=")[-1]
    transcript = get_youtube_transcript(video_id)
    chunks = split_text(transcript)
    retriever = build_retriever(chunks)
    rag_chain = create_rag_chain(retriever, model=model)
    summary = rag_chain.run("Summarize the video.")
    return summary

# Example usage
if __name__ == "__main__":
    video_url = "https://www.youtube.com/watch?v=VIDEO_ID"  # Replace with real video
    summary = summarize_youtube_video(video_url)
    print("\n=== Video Summary ===\n")
    print(summary)
